In [1]:
!pip install -q transformers==5.20.1 torch accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
print("Model loaded with plain transformers")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded with plain transformers


In [ ]:
#record the environment
import transformers, torch, bitsandbytes, datetime

env_info = f"""Session recorded at: {datetime.datetime.now()}
GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}
transformers: {transformers.__version__}
torch: {torch.__version__}
bitsandbytes: {bitsandbytes.__version__}
"""
with open("environment_03_structure.txt", "w") as f:
    f.write(env_info)
print(env_info)

Session recorded at: 2026-10-22 09:05:43.121719
GPU: Tesla T4
transformers: 5.20.1
torch: 2.11.0+cu130
bitsandbytes: 0.50.2



In [5]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
uploaded = files.upload()  # train_set.csv and val_set.csv

Mounted at /content/drive


Saving train_set.csv to train_set.csv
Saving val_set.csv to val_set.csv


In [6]:
import pandas as pd
import json
import numpy as np

train_df = pd.read_csv("train_set.csv")
val_df = pd.read_csv("val_set.csv")
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

pathology_columns = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Lesion', 'Lung Opacity', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]

path_to_pathology = {
    "lungs_and_airways.lung_opacity": "Lung Opacity",
    "lungs_and_airways.consolidation": "Consolidation",
    "lungs_and_airways.edema": "Edema",
    "lungs_and_airways.atelectasis": "Atelectasis",
    "lungs_and_airways.lung_lesion": "Lung Lesion",
    "lungs_and_airways.pneumonia": "Pneumonia",
    "pleura.pleural_effusion": "Pleural Effusion",
    "pleura.pneumothorax": "Pneumothorax",
    "pleura.pleural_other": "Pleural Other",
    "cardiovascular.cardiomegaly": "Cardiomegaly",
    "cardiovascular.enlarged_cardiomediastinum": "Enlarged Cardiomediastinum",
    "bones_and_chest_wall.fracture": "Fracture",
    "devices.support_devices": "Support Devices",
    "general.no_finding": "No Finding",
}

value_map = {
    "present": "1", "absent": "0",
    "uncertain": "-1", "not_mentioned": "NaN", "nan": "NaN"
}

def parse_structure_output(output_text):
    start = output_text.find("{")
    end = output_text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return ["NaN"] * 14
    json_text = output_text[start:end+1]
    try:
        data = json.loads(json_text)
    except:
        return ["NaN"] * 14

    labels = {}
    for path, pathology in path_to_pathology.items():
        keys = path.split(".")
        current = data
        for k in keys:
            if isinstance(current, dict) and k in current:
                current = current[k]
            else:
                current = None
                break
        if current is not None and isinstance(current, str):
            val = current.strip().lower()
            labels[pathology] = value_map.get(val, "NaN")
        else:
            labels[pathology] = "NaN"
    return [labels.get(p, "NaN") for p in pathology_columns]

def compute_metrics(results):
    metrics = {}
    for p_idx, pathology in enumerate(pathology_columns):
        tp = fp = fn = tn = 0
        for r in results:
            truth_val = r['truth'][p_idx]
            pred_val = r['predicted'][p_idx]
            if truth_val not in [0.0, 1.0]:
                continue
            try:
                pred_num = float(pred_val) if pred_val not in ["NaN", "nan"] else None
            except:
                pred_num = None
            if pred_num not in [0.0, 1.0]:
                continue
            truth_int = int(truth_val)
            pred_int = int(pred_num)
            if truth_int == 1 and pred_int == 1: tp += 1
            elif truth_int == 0 and pred_int == 1: fp += 1
            elif truth_int == 1 and pred_int == 0: fn += 1
            elif truth_int == 0 and pred_int == 0: tn += 1
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        metrics[pathology] = {'precision': precision, 'recall': recall, 'f1': f1,
                               'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}
    return metrics

Train: 2768, Val: 395


In [7]:
def build_zero_shot_structure_prompt(report_text):
    prompt = f"""You are a radiologist assistant. Analyze the chest X-ray report and output JSON only. Use: present, absent, uncertain, or not_mentioned for each finding.

Template:
{{"lungs_and_airways":{{"lung_opacity":"","consolidation":"","edema":"","atelectasis":"","lung_lesion":"","pneumonia":""}},"pleura":{{"pleural_effusion":"","pneumothorax":"","pleural_other":""}},"cardiovascular":{{"cardiomegaly":"","enlarged_cardiomediastinum":""}},"bones_and_chest_wall":{{"fracture":""}},"devices":{{"support_devices":""}},"general":{{"no_finding":""}}}}

Report:
{report_text}

JSON:"""
    return prompt

def build_few_shot_structure_prompt(report_text, examples):
    example_text = ""
    for i, (_, ex) in enumerate(examples.iterrows()):
        report_snippet = ex['full_report']
        labels_dict = {}
        for pathology in pathology_columns:
            val = ex[pathology]
            if pd.isna(val):
                word = "not_mentioned"
            elif val == 1.0:
                word = "present"
            elif val == 0.0:
                word = "absent"
            elif val == -1.0:
                word = "uncertain"
            else:
                word = "not_mentioned"
            labels_dict[pathology] = word

        example_json = {
            "lungs_and_airways": {
                "lung_opacity": labels_dict["Lung Opacity"],
                "consolidation": labels_dict["Consolidation"],
                "edema": labels_dict["Edema"],
                "atelectasis": labels_dict["Atelectasis"],
                "lung_lesion": labels_dict["Lung Lesion"],
                "pneumonia": labels_dict["Pneumonia"],
            },
            "pleura": {
                "pleural_effusion": labels_dict["Pleural Effusion"],
                "pneumothorax": labels_dict["Pneumothorax"],
                "pleural_other": labels_dict["Pleural Other"],
            },
            "cardiovascular": {
                "cardiomegaly": labels_dict["Cardiomegaly"],
                "enlarged_cardiomediastinum": labels_dict["Enlarged Cardiomediastinum"],
            },
            "bones_and_chest_wall": {"fracture": labels_dict["Fracture"]},
            "devices": {"support_devices": labels_dict["Support Devices"]},
            "general": {"no_finding": labels_dict["No Finding"]},
        }
        example_text += f"\nExample {i+1}:\nReport: {report_snippet}\nJSON: {json.dumps(example_json, separators=(',', ':'))}\n"

    prompt = f"""You are a radiologist assistant. Analyze the chest X-ray report and output JSON only. Use: present, absent, uncertain, or not_mentioned for each finding.

Template:
{{"lungs_and_airways":{{"lung_opacity":"","consolidation":"","edema":"","atelectasis":"","lung_lesion":"","pneumonia":""}},"pleura":{{"pleural_effusion":"","pneumothorax":"","pleural_other":""}},"cardiovascular":{{"cardiomegaly":"","enlarged_cardiomediastinum":""}},"bones_and_chest_wall":{{"fracture":""}},"devices":{{"support_devices":""}},"general":{{"no_finding":""}}}}
{example_text}
Now classify this report:
Report: {report_text}

JSON:"""
    return prompt

def run_structure_batch(df, prompt_fn, examples=None):
    results = []
    for i in range(len(df)):
        sample = df.iloc[i]
        report_text = sample['full_report']
        prompt = prompt_fn(report_text, examples) if examples is not None else prompt_fn(report_text)
        messages = [{"role": "user", "content": prompt}]
        formatted_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
        output = model.generate(
            **inputs, max_new_tokens=300,
            do_sample=False, repetition_penalty=1.1
        )
        generated_ids = output[0][inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True)
        predicted = parse_structure_output(raw)
        truth = sample[pathology_columns].values.tolist()
        results.append({'id': sample['id'], 'predicted': predicted, 'truth': truth, 'raw': raw})
        if (i + 1) % 50 == 0:
            print(f"Processed {i+1}/{len(df)}")
    return results

def get_random_examples(df, n, seed):
    return df.sample(n=n, replace=False, random_state=seed)

In [8]:
examples = get_random_examples(train_df, 10, 42)
prompt = build_few_shot_structure_prompt(val_df.iloc[0]['full_report'], examples)
print("prompt tokens:", len(tokenizer(prompt)["input_ids"]))
smoke = run_structure_batch(val_df.head(2), build_few_shot_structure_prompt, examples)
print(smoke[0]['raw'])

prompt tokens: 2427


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Here's the classification of the report:

```
{
  "lungs_and_airways": {
    "lung_opacity": "present",
    "consolidation": "not_mentioned",
    "edema": "present",
    "atelectasis": "uncertain",
    "lung_lesion": "uncertain",
    "pneumonia": "uncertain"
  },
  "pleura": {
    "pleural_effusion": "present",
    "pneumothorax": "not_mentioned",
    "pleural_other": "not_mentioned"
  },
  "cardiovascular": {
    "cardiomegaly": "not_mentioned",
    "enlarged_cardiomediastinum": "not_mentioned"
  },
  "bones_and_chest_wall": {
    "fracture": "not_mentioned"
  },
  "devices": {
    "support_devices": "not_mentioned"
  },
  "general": {
    "no_finding": "not_mentioned"
  }
}
```


**Structure zero-shot on the full validation set**

In [9]:
structure_zs_results = run_structure_batch(val_df, build_zero_shot_structure_prompt)
zs_metrics = compute_metrics(structure_zs_results)

print(f"{'Pathology':30s} | {'Precision':>9} | {'Recall':>6} | {'F1':>6}")
print("-" * 70)
for p, m in zs_metrics.items():
    print(f"{p:30s} | {m['precision']:>9.2f} | {m['recall']:>6.2f} | {m['f1']:>6.2f}")

zs_macro = np.mean([m['f1'] for m in zs_metrics.values()])
print(f"\nMacro F1 (structure zero-shot, transformers): {zs_macro:.3f}")

rows = []
for r in structure_zs_results:
    row = {'id': r['id'], 'raw': r['raw']}
    for i, p in enumerate(pathology_columns):
        row[f'truth_{p}'] = r['truth'][i]
        row[f'pred_{p}'] = r['predicted'][i]
    rows.append(row)
pd.DataFrame(rows).to_csv("/content/drive/MyDrive/structure_zero_shot_results.csv", index=False)
print("Saved structure_zero_shot_results.csv")

from google.colab import files
files.download("/content/drive/MyDrive/structure_zero_shot_results.csv")

Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Pathology                      | Precision | Recall |     F1
----------------------------------------------------------------------
No Finding                     |      1.00 |   0.98 |   0.99
Enlarged Cardiomediastinum     |      0.40 |   0.27 |   0.32
Cardiomegaly                   |      0.90 |   0.53 |   0.67
Lung Lesion                    |      1.00 |   1.00 |   1.00
Lung Opacity                   |      0.33 |   1.00 |   0.49
Edema                          |      0.80 |   1.00 |   0.89
Consolidation                  |      1.00 |   1.00 |   1.00
Pneumonia                      |      1.00 |   0.71 |   0.83
Atelectasis                    |      1.00 |   1.00 |   1.00
Pneumothorax                   |      1.00 |   0.33 |   0.50
Pleural Effusion               |      0.67 |   0.53 |   0.59
Pleural Other                  |      0.00 |   0.00 |   0.00
Fracture   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Structure few-shot, 5 seeds**

In [ ]:
seeds = [42, 123, 456, 789, 1011]
structure_fs_run_results = {}
structure_fs_macro_scores = []

for seed in seeds:
    print(f"\n=== Seed {seed} ===")
    examples = get_random_examples(train_df, 10, seed)
    results = run_structure_batch(val_df, build_few_shot_structure_prompt, examples)
    structure_fs_run_results[seed] = results

    rows = []
    for r in results:
        row = {'id': r['id'], 'raw': r['raw']}
        for i, p in enumerate(pathology_columns):
            row[f'truth_{p}'] = r['truth'][i]
            row[f'pred_{p}'] = r['predicted'][i]
        rows.append(row)
    pd.DataFrame(rows).to_csv(f"/content/drive/MyDrive/structure_few_shot_seed{seed}_results.csv", index=False)
    print(f"Saved structure_few_shot_seed{seed}_results.csv")
    files.download(f"/content/drive/MyDrive/structure_few_shot_seed{seed}_results.csv")

    metrics = compute_metrics(results)
    macro = np.mean([m['f1'] for m in metrics.values()])
    structure_fs_macro_scores.append(macro)
    print(f"Seed {seed}: Macro F1 = {macro:.3f}")

print(f"\n=== Aggregate ===")
print(f"All runs: {[f'{s:.3f}' for s in structure_fs_macro_scores]}")
print(f"Mean ± Std: {np.mean(structure_fs_macro_scores):.3f} ± {np.std(structure_fs_macro_scores):.3f}")


=== Seed 42 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Saved structure_few_shot_seed42_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Seed 42: Macro F1 = 0.780

=== Seed 123 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Saved structure_few_shot_seed123_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Seed 123: Macro F1 = 0.759

=== Seed 456 ===
Processed 50/395
Processed 100/395
Processed 150/395
Processed 200/395
Processed 250/395
Processed 300/395
Processed 350/395
Saved structure_few_shot_seed456_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Seed 456: Macro F1 = 0.820

=== Seed 789 ===
Processed 50/395
Processed 100/395
Processed 150/395


In [ ]:
def count_invalid(results):
    invalid = 0
    for r in results:
        raw = r.get('raw', '')
        start = raw.find("{")
        end = raw.rfind("}")
        if start == -1 or end == -1 or end <= start:
            invalid += 1
            continue
        try:
            json.loads(raw[start:end+1])
        except:
            invalid += 1
            continue
        # count all-NaN predictions as unusable
        if all(v == 'NaN' for v in r['predicted']):
            invalid += 1
    return invalid

print(f"Zero-shot invalid JSON: {count_invalid(structure_zs_results)} / {len(structure_zs_results)}")
for seed, results in structure_fs_run_results.items():
    print(f"Seed {seed} invalid JSON: {count_invalid(results)} / {len(results)}")